# Lab 5 essentials: give the model tools, and check what it does with them

**Time: about 50 minutes. Click Run all, then read each step while it runs.**

**What you will do.** Until now the model only wrote text. In this lab it gets three **tools**: one
searches FreshCart's policy documents, one returns an official business number, and one runs a SQL
query on the warehouse. For each question the model decides which tool to use and what to send it.
Your code runs the tool and hands the result back, and the model writes the answer. At the end you
put two of the tools behind an **MCP server**, so any AI app could use them. Along the way you will
find three problems that no error message tells you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| `pypdf`, `docx2txt` | read the PDF and Word documents, as in Lab 1 | 1 | [pypdf.readthedocs.io](https://pypdf.readthedocs.io/) |
| `langchain-community` | `BM25Retriever`: the keyword search from Lab 3 | 1 | [python.langchain.com](https://python.langchain.com/docs/integrations/retrievers/bm25/) |
| `sqlite3` (built into Python) | runs SQL on the FreshCart warehouse, a single file | 1, 2 | [docs.python.org](https://docs.python.org/3/library/sqlite3.html) |
| `langchain-core` | `@tool`: turns a Python function into a tool the model can ask for; the message types | 2, 6 | [python.langchain.com](https://python.langchain.com/docs/concepts/tools/) |
| `langchain-openai` | `ChatOpenAI`: talks to the model. Databricks serves its models in the same format as OpenAI, so this works with a Databricks model too | 3 | [python.langchain.com](https://python.langchain.com/docs/integrations/chat/openai/) |
| `databricks-sdk` (already on Databricks) | `WorkspaceClient`: signs you in to this workspace, so there is no key to copy | 3 | [docs.databricks.com](https://docs.databricks.com/aws/en/dev-tools/sdk-python) |
| `mcp`, `langchain-mcp-adapters` | build an MCP server, and load its tools back into LangChain | 8 | [modelcontextprotocol.io](https://modelcontextprotocol.io/) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell.

**Five words used throughout.**

- A **tool** is a Python function the model may ask your code to run. It has a name, a description
  and a list of inputs. The model never sees the code inside.
- A **tool call** is the model's request: the tool's name plus the inputs it chose. The model does
  not run anything itself; your code decides whether to run it.
- The **tool description** is the text that tells the model when to use the tool. It is the only
  thing the model reads about it.
- The **tool loop**: ask the model, run the tools it asks for, send back the results, and repeat
  until the model answers in words.
- **MCP** (Model Context Protocol) is a standard way for a program to offer tools to any AI app. You
  write the tools once, and Claude, Copilot or your own LangChain code can all use them.

**Cost.** The model is served by Databricks and billed per use: this notebook makes about 50 short
calls, a few cents in total.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above, then restarts Python. It takes about
one minute. The restart is expected. A red line about `fastapi` and `starlette` can be ignored: this
lab does not use either.

In [ ]:
# Loaders and keyword search from Labs 1 and 3, the model connector, and the two MCP libraries.
%pip install -q langchain-core langchain-community langchain-openai langchain-mcp-adapters mcp pypdf docx2txt rank_bm25

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
dbutils.library.restartPython()

In [ ]:
# warnings controls which notices are shown.
import warnings

# Hide notices that are not errors, such as messages about packages being renamed.
warnings.filterwarnings("ignore")
print("ready")

## Step 1. The two things the tools will read: documents and the warehouse

**What this does.** Copies two things out of the course zip file into this session's own folder:

- the **warehouse**, `freshcart.db`: a SQLite file with orders, customers, refund claims and
  restaurants. SQLite keeps a whole database in one file, so there is no server to start.
- three **policy documents**: the 2024 refund policy, the partner agreement and the support SLA
  matrix. They are cut into pieces and indexed for keyword search, as in Lab 3.

**What you should see.** `10 pieces indexed`, then the warehouse's eight tables with their row counts,
for example `orders_v2: 5988 rows`. Two tables, `orders` and `order_items`, are traps from Lab 5: an
old copy of the orders, and one row per item, which counts an order several times.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# zipfile reads the course zip file; pathlib handles file paths; sqlite3 opens the warehouse.
import zipfile, pathlib, sqlite3
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

# The course zip file, and a folder of our own for this lab.
PACK = PACK_ZIP
HOME = pathlib.Path("/tmp/lab05_essentials")
(HOME / "docs").mkdir(parents=True, exist_ok=True)

# Copy the warehouse and the three documents out of the zip file.
with zipfile.ZipFile(PACK) as z:
    (HOME / "freshcart.db").write_bytes(z.read("data/freshcart.db"))
    for name in ["refund_policy_v3_2024.pdf", "partner_agreement_extract.pdf", "support_sla_matrix.docx"]:
        (HOME / "docs" / name).write_bytes(z.read("data/raw/" + name))

# Read the documents, cut them into pieces of about 500 characters, and build the keyword search.
docs = (PyPDFLoader(str(HOME / "docs" / "refund_policy_v3_2024.pdf")).load()
        + PyPDFLoader(str(HOME / "docs" / "partner_agreement_extract.pdf")).load()
        + Docx2txtLoader(str(HOME / "docs" / "support_sla_matrix.docx")).load())
pieces = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
# k=3: each search returns the 3 best-matching pieces.
keyword_search = BM25Retriever.from_documents(pieces, k=3)
print(len(pieces), "pieces indexed")

# Where the warehouse is. mode=ro opens it read-only: even a bad query cannot change it.
DB = HOME / "freshcart.db"
def query(sql):
    connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    try:
        # Return at most 50 rows, so a careless query cannot flood the model.
        return connection.execute(sql).fetchmany(50)
    finally:
        connection.close()

# List each table and how many rows it holds.
for (table,) in query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
    print(f"  {table}: {query(f'SELECT COUNT(*) FROM {table}')[0][0]} rows")

## Step 2. Three Python functions become three tools

**What this does.** The `@tool` line above a function turns it into a tool. LangChain reads three
things from the function and nothing else: its **name**, its **description** (the text in triple
quotes) and its **inputs** with their types. That is what the model will see.

- `search_documents(question)`: the keyword search from Step 1.
- `get_metric(metric, start_date, end_date)`: one **official** business number. The rules are
  written into the code once: revenue counts delivered orders only, and test accounts are always left
  out. `Literal[...]` lists the only three metric names allowed.
- `run_sql(sql)`: any read-only query on the warehouse. A check in code refuses anything that could
  change data, before the query reaches the database.

The last part of the cell prints `get_metric` exactly as the model receives it. This format, a name,
a description and a list of typed inputs, is the same one **OpenAPI** uses to describe a web API, and
the Lab 4 API's `/docs` page shows its own in this format.

**What you should see.** The description of `get_metric`, with `metric` limited to three names. Then
three tools called by hand, with no model involved: revenue for March 2024 is `531043.03`, orders in
May 2024 are `450`, and a `DELETE` is refused.

In [ ]:
# re finds patterns in text; json prints the tool description neatly; Literal lists allowed values.
import re, json
from typing import Literal
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def search_documents(question: str) -> str:
    """Search FreshCart's policy documents (refund policy 2024, partner agreement, support SLA matrix) and return the 3 most relevant passages. Use for questions about rules, policies, time limits and definitions."""
    # Join the 3 best pieces into one text, with a line between them.
    return "\n---\n".join(p.page_content for p in keyword_search.invoke(question))

@tool
def get_metric(metric: Literal["revenue", "order_count", "refund_claims"], start_date: str, end_date: str) -> float:
    """Return one official FreshCart business number for a date range. revenue: delivered orders only. order_count: all orders. refund_claims: number of claims. Test accounts are always excluded. Dates are YYYY-MM-DD, both ends included."""
    # Each metric is written once, with its rules: test accounts out, and revenue only for delivered orders.
    orders = "FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id WHERE c.is_internal = 0"
    sql = {
        "revenue": f"SELECT SUM(o.order_value) {orders} AND o.status = 'delivered' AND o.order_date BETWEEN ? AND ?",
        "order_count": f"SELECT COUNT(*) {orders} AND o.order_date BETWEEN ? AND ?",
        "refund_claims": "SELECT COUNT(*) FROM fact_refund_claim f JOIN dim_customer c ON c.customer_id = f.customer_id "
                         "WHERE c.is_internal = 0 AND f.claim_date BETWEEN ? AND ?",
    }[metric]
    # The ? marks are filled with the dates by sqlite3 itself, so text in a date cannot change the query.
    connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    value = connection.execute(sql, (start_date, end_date)).fetchone()[0]
    connection.close()
    return round(value or 0, 2)

# Words that change data. A query containing any of them is refused.
CHANGES_DATA = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b", re.IGNORECASE)

@tool
def run_sql(sql: str) -> str:
    """Run one read-only SQLite SELECT on the FreshCart warehouse and return up to 50 rows. Tables: orders_v2(order_id, customer_id, restaurant_id, order_date, order_value, delivery_minutes, status), dim_customer(customer_id, name, city, region, signup_date, status, is_internal), fact_refund_claim(claim_id, order_id, customer_id, claim_date, reason, amount, status), dim_restaurant(restaurant_id, name, city, cuisine, partner_since)."""
    statement = sql.strip().rstrip(";")
    # The check, in code: one statement, starting with SELECT, with no word that changes data.
    if ";" in statement or not statement.lower().startswith("select") or CHANGES_DATA.search(statement):
        return "REFUSED: only one read-only SELECT is allowed"
    return str(query(statement))

TOOLS = [search_documents, get_metric, run_sql]

# What the model receives for get_metric: name, description and typed inputs, and no code.
print(json.dumps(convert_to_openai_tool(get_metric), indent=1))

# Call each tool by hand, the way your code will when the model asks.
print("\nrevenue, March 2024:", get_metric.invoke({"metric": "revenue", "start_date": "2024-03-01", "end_date": "2024-03-31"}))
print("orders, May 2024   :", get_metric.invoke({"metric": "order_count", "start_date": "2024-05-01", "end_date": "2024-05-31"}))
print("a DELETE           :", run_sql.invoke({"sql": "DELETE FROM dim_customer"}))

## Step 3. Give the tools to the model, and see which one it picks

**What this does.** Connects to `databricks-meta-llama-3-3-70b-instruct`, a language model this
workspace serves. `WorkspaceClient` signs in as you and hands over a short-lived sign-in token, which
the code passes on without printing it. `bind_tools` attaches the three tool descriptions to every
question. Then 10 questions go to the model, and for each one the cell prints the tool it asked for
and the inputs it chose. **Nothing is run yet**: this step only looks at the model's choices.

The 10 questions each have one right tool: 3 for the documents, 3 for official numbers, 3 that need
their own SQL, and one, about France, that needs no tool at all.

**What you should see.** `right tool: 10/10` or `9/10`: the test runs gave each once. When it is
9, the miss is the May orders question sent to `run_sql` instead of `get_metric`. Both tools could
answer it, and Step 7 shows why that choice matters: the two give different numbers. Read the inputs
too: the model turned "March 2024" into `2024-03-01` and `2024-03-31` by itself, because the
description asked for YYYY-MM-DD, and it wrote its own SQL for the city question.

**Try this.** Add "Always use this instead of run_sql for revenue, order counts and claim counts." to
the end of `get_metric`'s description in Step 2, run Steps 2 and 3 again, and see if the miss goes.

In [ ]:
from databricks.sdk import WorkspaceClient
from langchain_openai import ChatOpenAI

# Sign in as you. The token goes straight into the connector and is never printed.
workspace = WorkspaceClient()
token = workspace.config.authenticate()["Authorization"].split(" ", 1)[1]

# The served model, reached at this workspace's address. temperature=0 asks for the most likely reply.
llm = ChatOpenAI(model="databricks-meta-llama-3-3-70b-instruct",
                 base_url=workspace.config.host.rstrip("/") + "/serving-endpoints",
                 api_key=token, temperature=0, max_retries=5)

# The same model, with the three tool descriptions sent along with every question.
model = llm.bind_tools(TOOLS)

# Ten questions and the tool each one should use ("none": the model should just answer).
QUESTIONS = [
    ("How many refund claims can a customer make in thirty days?", "search_documents"),
    ("What was revenue in March 2024?", "get_metric"),
    ("How many orders were placed in May 2024?", "get_metric"),
    ("Which city had the most orders in 2024?", "run_sql"),
    ("When is wallet credit applied?", "search_documents"),
    ("How many refund claims were filed in March 2024?", "get_metric"),
    ("How many restaurants partner with FreshCart?", "run_sql"),
    ("What was the average delivery time in minutes in June 2024?", "run_sql"),
    ("What does spillage mean under the refund policy?", "search_documents"),
    ("What is the capital of France?", "none"),
]

# Ask every question, print the model's choice, and count how many picked the right tool.
def check_choices(model):
    right = 0
    for question, want in QUESTIONS:
        reply = model.invoke(question)
        # reply.tool_calls is the list of tools the model asked for; empty means it answered in words.
        picked = reply.tool_calls[0]["name"] if reply.tool_calls else "none"
        right += picked == want
        inputs = reply.tool_calls[0]["args"] if reply.tool_calls else reply.content[:60]
        print(f"{'ok  ' if picked == want else 'MISS'} {question[:52]:52} -> {picked:16} {inputs}")
    print(f"\nright tool: {right}/{len(QUESTIONS)}")

check_choices(model)

## Step 4. Problem 1: the model only knows what the description says

**What this does.** Makes the same three tools again, with the same code inside, but with one-line
descriptions: "Search.", "Get a number.", "Run SQL." The `metric` input also loses its list of allowed
names. Then it asks the same 10 questions.

**What you should see.** Close to `right tool: 6/10`, the same in both test runs. Look at how the
misses fail, because none of them raises an error:

- The May orders and March claims questions go to `run_sql`, with SQL on tables that do not exist
  (`orders`, `refund_claims`): the model no longer knows the table names.
- The restaurant question goes to `search_documents`.
- The delivery time question goes to `get_metric` with a metric the model **made up**,
  `average_delivery_time`. With the list of allowed names gone, nothing stopped it.

**Why this matters.** The description is the whole contract between your code and the model. Writing
it well is part of building the tool, like naming the inputs.

In [ ]:
# The same three functions, with descriptions too short to choose by.
@tool("search_documents")
def vague_search(question: str) -> str:
    """Search."""
    return search_documents.invoke({"question": question})

# metric is now any text, not one of three names.
@tool("get_metric")
def vague_metric(metric: str, start_date: str, end_date: str) -> float:
    """Get a number."""
    return get_metric.invoke({"metric": metric, "start_date": start_date, "end_date": end_date})

@tool("run_sql")
def vague_sql(sql: str) -> str:
    """Run SQL."""
    return run_sql.invoke({"sql": sql})

# Same model, same questions; only the descriptions changed.
check_choices(llm.bind_tools([vague_search, vague_metric, vague_sql]))

## Step 5. Problem 2: the model will ask to change data

**What this does.** Sends three requests that would change the warehouse, the kind a user might type
without thinking. For each one it prints the tool call the model made, then what the check in
`run_sql` did with it.

**What you should see.** The model asks for a `DELETE` twice and an `UPDATE` once: 3 of 3, with no
warning. All 3 are refused by the check, so nothing reaches the database. The model did not protect
the data. The check in code did, and the warehouse was also opened read-only (Step 1) in case the
check missed something.

In [ ]:
# Three requests that would change data.
risky = ["Delete the test accounts from dim_customer.",
         "Remove customer 1 from the warehouse, it is a test account.",
         "Set every order in May 2024 to delivered."]

for request in risky:
    reply = model.invoke(request)
    for call in reply.tool_calls:
        print(request)
        print("   model asked for:", call["name"], call["args"])
        # Run the tool the way the tool loop would, so the check decides.
        print("   the check said :", run_sql.invoke(call["args"]) if call["name"] == "run_sql" else "(not run_sql)")

## Step 6. The tool loop: from question to answer

**What this does.** Writes the loop that turns tool calls into answers, in one function, `ask`:

1. send the question to the model;
2. if the model asks for tools, run each one and add its result to the conversation as a
   `ToolMessage`, matched to the request by an id;
3. send the whole conversation back and repeat, up to 4 times, until the model answers in words.

`ask` is written with `async` and `await` because the MCP tools in Step 8 only work that way.
`await` simply waits for the result; the rest reads like ordinary Python.

**What you should see.** Four answers, each with the tool calls printed above it. The comparison
question makes **two** calls, one for March and one for February, and the model works out the
difference itself: 531,043.03 against 522,100.56, about 1.7% higher.

Look at the currency: the model writes a dollar sign. Nothing in the warehouse says dollars, and
FreshCart's cities are in India. The model added it, which is one reason Lab 6 builds the sentence
from the number in code instead.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# Find a tool by its name, because a tool call only gives the name.
def tools_by_name(tools):
    return {t.name: t for t in tools}

async def ask(question, model=model, tools=TOOLS, rounds=4):
    by_name = tools_by_name(tools)
    conversation = [HumanMessage(question)]
    for _ in range(rounds):
        reply = await model.ainvoke(conversation)
        conversation.append(reply)
        # No tool calls means the model answered in words: we are done.
        if not reply.tool_calls:
            return reply.content
        for call in reply.tool_calls:
            # Run the tool the model asked for, with the inputs it chose.
            result = await by_name[call["name"]].ainvoke(call["args"])
            print(f"   tool call: {call['name']} {call['args']} -> {str(result)[:90]}")
            # Send the result back, labelled with the id of the request it answers.
            conversation.append(ToolMessage(str(result), tool_call_id=call["id"]))
    return "(no answer after 4 rounds)"

for question in ["What was revenue in March 2024?",
                 "What was revenue in March 2024, and how did it compare with February 2024?",
                 "How many refund claims can a customer make in thirty days?",
                 "Which city had the most orders in 2024?"]:
    print(question)
    print("=>", await ask(question), "\n")

## Step 7. Problem 3: take away the official numbers, and the numbers change

**What this does.** Gives the model only two tools, the documents and `run_sql`, so it has to write
its own SQL for the same two questions. The answer comes back in a normal, confident sentence.

**What you should see.** In both test runs:

| question | with `get_metric` | with its own SQL |
|---|---|---|
| revenue, March 2024 | 531,043.03 | 590,657.18 |
| orders, May 2024 | 450 | 491 |

The SQL is not wrong as SQL. It adds every order, including the undelivered ones and those from
FreshCart's two test accounts, because nothing told the model the business rules. No error, and a
number 11% too high. That is the problem Lab 6 is about.

In [ ]:
# Only the documents and raw SQL: no official numbers.
two_tools = [search_documents, run_sql]
sql_only_model = llm.bind_tools(two_tools)

for question in ["What was revenue in March 2024?", "How many orders were placed in May 2024?"]:
    print(question)
    print("=>", await ask(question, model=sql_only_model, tools=two_tools), "\n")

## Step 8. The same tools as an MCP server

**What this does.** In a real company, the data team owns the warehouse tools and many apps want to
use them. MCP lets the data team publish the tools once as a **server**: a small program that lists
its tools and runs them when asked.

1. The first part writes the server, `freshcart_server.py`. It holds the same `get_metric` and
   `run_sql` code as Step 2. `@server.tool()` plays the part of `@tool`.
2. `MultiServerMCPClient` starts that program and asks it for its tools. They come back as ordinary
   LangChain tools, with the names and descriptions the server gave them.
3. The model gets those two tools plus our own `search_documents`, and answers through the same `ask`.

The server talks over **stdio**, its standard input and output, so it runs on this machine with no
network. A real one would usually run on a server, and Claude Desktop, Copilot Studio or any other MCP
app could connect to it the same way.

**What you should see.** `tools from the MCP server: ['get_metric', 'run_sql']`, then the March
revenue answer, 531,043.03, the same as Step 6. The server sends its result back as text with a
label, `'type': 'text'`, which is the MCP format. The model cannot tell where a tool runs.

In [ ]:
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

# The server program, written to a file. It repeats Step 2's code, because it runs on its own.
(HOME / "freshcart_server.py").write_text(f"""
import re, sqlite3
from typing import Literal
from mcp.server.fastmcp import FastMCP

DB = "{DB}"
server = FastMCP("freshcart-warehouse")

def query(sql, params=()):
    connection = sqlite3.connect(f"file:{{DB}}?mode=ro", uri=True)
    try:
        return connection.execute(sql, params).fetchmany(50)
    finally:
        connection.close()

@server.tool()
def get_metric(metric: Literal["revenue", "order_count", "refund_claims"], start_date: str, end_date: str) -> float:
    {get_metric.description!r}
    orders = "FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id WHERE c.is_internal = 0"
    sql = {{
        "revenue": f"SELECT SUM(o.order_value) {{orders}} AND o.status = 'delivered' AND o.order_date BETWEEN ? AND ?",
        "order_count": f"SELECT COUNT(*) {{orders}} AND o.order_date BETWEEN ? AND ?",
        "refund_claims": "SELECT COUNT(*) FROM fact_refund_claim f JOIN dim_customer c ON c.customer_id = f.customer_id "
                         "WHERE c.is_internal = 0 AND f.claim_date BETWEEN ? AND ?",
    }}[metric]
    return round(query(sql, (start_date, end_date))[0][0] or 0, 2)

CHANGES_DATA = re.compile(r"\\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\\b", re.IGNORECASE)

@server.tool()
def run_sql(sql: str) -> str:
    {run_sql.description!r}
    statement = sql.strip().rstrip(";")
    if ";" in statement or not statement.lower().startswith("select") or CHANGES_DATA.search(statement):
        return "REFUSED: only one read-only SELECT is allowed"
    return str(query(statement))

if __name__ == "__main__":
    server.run(transport="stdio")
""")

# Start the server program and ask it for its tools. sys.executable is this notebook's own Python.
client = MultiServerMCPClient({"freshcart": {"command": sys.executable,
                                             "args": [str(HOME / "freshcart_server.py")],
                                             "transport": "stdio"}})
server_tools = await client.get_tools()
print("tools from the MCP server:", [t.name for t in server_tools])

# The server's two tools plus our own document search, given to the same model and the same loop.
mixed_tools = server_tools + [search_documents]
question = "What was revenue in March 2024?"
print(question)
print("=>", await ask(question, model=llm.bind_tools(mixed_tools), tools=mixed_tools))

## What you found

1. **The model picks tools well when the descriptions are good** (Step 3): 9 or 10 of 10, including
   the question that needed no tool. The one miss was a question two tools could answer.
2. **Vague descriptions break it silently** (Step 4): 6 of 10, with SQL on tables that do not exist
   and a made-up metric name. The list of allowed names in `Literal[...]` is a check too.
3. **The model will ask to change data** (Step 5): 3 of 3 risky requests became a `DELETE` or an
   `UPDATE`. The check in code, and the read-only connection, are what stopped them.
4. **Official numbers need an official tool** (Step 7): the model's own SQL gave 590,657.18 and 491
   instead of 531,043.03 and 450, with no error.
5. **MCP moves a tool out of the notebook without changing it** (Step 8): same tool, same answer, now
   usable by any MCP app.

**Where this goes next.** Lab 6 essentials takes Problem 3 apart: why the model's SQL gives 491, how
business rules in the prompt help, and how a semantic layer (the idea behind `get_metric`) removes
the problem.

## Look it up yourself: every setting the libraries offer

**What this does.** For the main things this lab calls, the cell prints every parameter (read from the
installed library), the values this lab passed, and one setting worth trying.

**What you should see.** Four short blocks.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap

# parameters(thing): the parameters `thing` accepts, as text such as "temperature=None".
def parameters(thing):
    # Some classes list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        return [n if f.is_required() else f"{n}={f.get_default(call_default_factory=True)!r}"
                for n, f in thing.model_fields.items()]
    try:
        sig = inspect.signature(thing)
    except (TypeError, ValueError):
        return []
    # Skip "self", "cls", and the *args / **kwargs catch-alls, which are not named settings.
    return [p.name if p.default is p.empty else f"{p.name}={p.default!r}" for p in sig.parameters.values()
            if p.name not in ("self", "cls") and p.kind not in (p.VAR_POSITIONAL, p.VAR_KEYWORD)]

# look_up prints the three lines for one function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    print(name)
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

look_up("tool", tool, "a function, with its description in triple quotes",
        "a name in brackets, @tool(\"name\"), as Step 4 does, to name the tool differently from the function")
look_up("ChatOpenAI.bind_tools", ChatOpenAI.bind_tools, "TOOLS",
        "tool_choice: force one tool, or \"any\" to make the model use some tool")
look_up("ChatOpenAI", ChatOpenAI, "model, base_url (listed as openai_api_base), api_key (openai_api_key), temperature=0, max_retries=5",
        "max_tokens: caps the length of each reply, and so its cost")
look_up("MultiServerMCPClient", MultiServerMCPClient, '{"freshcart": {command, args, transport="stdio"}}',
        "more than one server in the same dictionary: tools from all of them in one list")

## Your notes

Double click to type: which of the three problems would hurt FreshCart most if it went unnoticed, and
what check you would add for it.